In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import operator

# Load the datasets
#path1 = r'c://DESKTOP-GFUFSM3/Users/wwwke/3D%20Objects/Unsupervised%20Learning/Kaggle%20Project/train1.csv/train.csv'

path_train = r'C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project\train.csv'
path_submission = r'C:\Users\wwwke\3D Objects\Unsupervised Learning\Kaggle Project\sample_submission.csv'

train = pd.read_csv(path_train)
test = pd.read_csv('test.csv')
sample_submission = pd.read_csv(path_submission)

# Inspect the data
print(train.head())
print(train.info())

# Check for missing values
print(train.isnull().sum())

# Verify data types
print(train.dtypes)

# Convert data types if necessary
train['userId'] = train['userId'].astype(int)
train['movieId'] = train['movieId'].astype(int)
train['rating'] = train['rating'].astype(float)

# Create the utility matrix
try:
    util_matrix = train.pivot_table(index='userId', columns='movieId', values='rating')
    print(util_matrix.shape)
except Exception as e:
    print(f"An error occurred: {e}")

# Normalize each row (a given user's ratings) of the utility matrix
util_matrix_norm = util_matrix.apply(lambda x: (x - np.mean(x)) / (np.max(x) - np.min(x)), axis=1)
# Fill NaN values with 0's, transpose matrix, and drop users with no ratings
util_matrix_norm.fillna(0, inplace=True)
util_matrix_norm = util_matrix_norm.T
util_matrix_norm = util_matrix_norm.loc[:, (util_matrix_norm != 0).any(axis=0)]
# Save the utility matrix in scipy's sparse matrix format
util_matrix_sparse = sp.csr_matrix(util_matrix_norm.values)

# Compute the similarity matrix using the cosine similarity metric
user_similarity = cosine_similarity(util_matrix_sparse.T)
# Save the matrix as a dataframe to allow for easier indexing
user_sim_df = pd.DataFrame(user_similarity, index=util_matrix_norm.columns, columns=util_matrix_norm.columns)

def collab_generate_top_N_recommendations(user, N=10, k=20):
    if user not in user_sim_df.columns:
        return train.groupby('movieId').mean().sort_values(by='rating', ascending=False).index[:N].to_list()

    sim_users = user_sim_df.sort_values(by=user, ascending=False).index[1:k+1]
    favorite_user_items = []
    most_common_favorites = {}

    for i in sim_users:
        max_score = util_matrix_norm.loc[:, i].max()
        favorite_user_items.append(util_matrix_norm[util_matrix_norm.loc[:, i] == max_score].index.tolist())

    for item_collection in range(len(favorite_user_items)):
        for item in favorite_user_items[item_collection]:
            if item in most_common_favorites:
                most_common_favorites[item] += 1
            else:
                most_common_favorites[item] = 1

    sorted_list = sorted(most_common_favorites.items(), key=operator.itemgetter(1), reverse=True)[:N]
    top_N = [x[0] for x in sorted_list]
    return top_N

def collab_generate_rating_estimate(movie_title, user, k=20, threshold=0.0):
    sim_users = user_sim_df.sort_values(by=user, ascending=False).index[1:k+1]
    user_values = user_sim_df.sort_values(by=user, ascending=False).loc[:, user].tolist()[1:k+1]
    rating_list = []
    weight_list = []

    for sim_idx, user_id in enumerate(sim_users):
        rating = util_matrix.loc[user_id, movie_title]
        similarity = user_values[sim_idx]

        if (np.isnan(rating)) or (similarity < threshold):
            continue
        elif not np.isnan(rating):
            rating_list.append(rating * similarity)
            weight_list.append(similarity)
    try:
        predicted_rating = sum(rating_list) / sum(weight_list)
    except ZeroDivisionError:
        predicted_rating = np.mean(util_matrix[movie_title])
    return predicted_rating

def generate_predictions():
    predictions = []
    for idx, row in test.iterrows():
        user = row['userId']
        movie = row['movieId']
        predicted_rating = collab_generate_rating_estimate(movie, user)
        predictions.append((f"{user}_{movie}", predicted_rating))
    return predictions

predictions = generate_predictions()
submission_df = pd.DataFrame(predictions, columns=['Id', 'rating'])
submission_df.to_csv('submission.csv', index=False)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

# Assuming you have a validation set to compare the true ratings with predicted ratings
y_true = validation_set['rating'].values
y_pred = [collab_generate_rating_estimate(movie, user) for user, movie in zip(validation_set['userId'], validation_set['movieId'])]
print(f"RMSE: {rmse(y_true, y_pred)}")

   userId  movieId  rating   timestamp
0    5163    57669     4.0  1518349992
1  106343        5     4.5  1206238739
2  146790     5459     5.0  1076215539
3  106362    32296     2.0  1423042565
4    9041      366     3.0   833375837
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000038 entries, 0 to 10000037
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 305.2 MB
None
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object


C:\Users\wwwke\anaconda3\envs\Python_Env\Lib\site-packages\pandas\core\reshape\reshape.py:143: RuntimeWarning: overflow encountered in scalar multiply
  num_cells = num_rows * num_columns


An error occurred: negative dimensions are not allowed


NameError: name 'util_matrix' is not defined

In [ ]:
# Inspect the data
print(train.head())
print(train.info())

# Check for missing values
print(train.isnull().sum())

# Verify data types
print(train.dtypes)

# Convert data types if necessary
train['userId'] = train['userId'].astype(int)
train['movieId'] = train['movieId'].astype(int)
train['rating'] = train['rating'].astype(float)